# Step 1: Dataset Exploration

In [1]:
from pathlib import Path

current_folder = Path.cwd().resolve()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

raw_data_folder = project_root / "data" / "raw"

print("Project root:", project_root)
print("Raw data folder:", raw_data_folder)

dataset_files = [file for file in raw_data_folder.iterdir() if file.is_file()]

print("\nFiles found in data/raw:")
for file in dataset_files:
    print("-", file.name)

Project root: C:\Users\Jiya\Documents\Malicious URL Detection
Raw data folder: C:\Users\Jiya\Documents\Malicious URL Detection\data\raw

Files found in data/raw:
- raw_data.csv


In [2]:
import pandas as pd

dataset_path = raw_data_folder / "raw_data.csv"

df = pd.read_csv(dataset_path, low_memory=False)

print("Dataset loaded successfully.")
print("Dataset size:", df.shape[0], "rows and", df.shape[1], "columns")

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully.
Dataset size: 253098 rows and 2 columns

Column names:
['url', 'label']

First 5 rows:


,url,label
0,https://www.visitcanada.com,legitimate
1,http://218.228.19.9/~yossi/9ssfpkz,phishing
2,https://www.msupress.msu.edu/series.php?series...,legitimate
3,https://docs.google.com/presentation/d/e/2PACX...,phishing
4,https://www.c250.columbia.edu/c250_celebrates/...,legitimate


## Dataset Quality Checks

In [3]:
expected_columns = {"url", "label"}
actual_columns = set(df.columns)

print("Expected columns present:", expected_columns.issubset(actual_columns))
print("Missing expected columns:", expected_columns - actual_columns)

print("\nData types:")
display(df.dtypes.to_frame(name="data_type"))

print("\nDataset dimensions:")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

Expected columns present: True
Missing expected columns: set()

Data types:


,data_type
url,object
label,object



Dataset dimensions:
Rows: 253,098
Columns: 2


In [4]:
url_missing = df["url"].isna().sum()
label_missing = df["label"].isna().sum()

url_empty = df["url"].fillna("").astype(str).str.strip().eq("").sum()
label_empty = df["label"].fillna("").astype(str).str.strip().eq("").sum()

print("Missing values")
print(f"URL missing values: {url_missing:,}")
print(f"Label missing values: {label_missing:,}")

print("\nEmpty or whitespace-only values")
print(f"URL empty values: {url_empty:,}")
print(f"Label empty values: {label_empty:,}")

Missing values
URL missing values: 0
Label missing values: 0

Empty or whitespace-only values
URL empty values: 0
Label empty values: 0


In [5]:
label_counts = df["label"].value_counts(dropna=False)

print("Unique label values:")
print(label_counts)

print("\nClass distribution percentages:")
print((df["label"].value_counts(normalize=True, dropna=False) * 100).round(2))

Unique label values:
label
legitimate    129420
phishing      123678
Name: count, dtype: int64

Class distribution percentages:
label
legitimate    51.13
phishing      48.87
Name: proportion, dtype: float64


In [6]:
exact_duplicate_rows = df.duplicated().sum()
duplicate_urls = df["url"].duplicated().sum()

url_label_counts = df.groupby("url")["label"].nunique()
conflicting_label_urls = (url_label_counts > 1).sum()

print(f"Exact duplicate rows: {exact_duplicate_rows:,}")
print(f"Duplicate URLs, regardless of label: {duplicate_urls:,}")
print(f"URLs assigned more than one different label: {conflicting_label_urls:,}")

Exact duplicate rows: 0
Duplicate URLs, regardless of label: 0
URLs assigned more than one different label: 0


In [8]:
from urllib.parse import urlsplit

planned_features = [
    "url_length",
    "num_dots",
    "num_slashes",
    "num_hyphens",
    "num_underscores",
    "num_special_characters",
    "num_digits",
    "digit_to_letter_ratio",
    "path_length",
    "query_string_length",
    "num_query_parameters",
    "num_at_symbols",
    "num_double_slash_occurrences",
    "url_entropy",
    "domain_length",
    "num_subdomains",
    "tld_type",
    "suspicious_tld",
    "punycode_idn_usage",
    "ip_address_instead_of_domain",
    "https_usage",
    "url_shortening_service",
    "encoded_characters",
    "hexadecimal_characters",
    "suspicious_keywords",
    "brand_name_presence",
    "excessive_hyphens",
    "random_looking_domain"
]

print(f"Planned feature count: {len(planned_features)}")

url_series = df["url"].dropna().astype(str).str.strip()
non_empty_urls = url_series[url_series.ne("")]

# A URL is structurally usable here when it has both a scheme and a domain/IP part.
def has_scheme_and_netloc(url):
    try:
        parsed = urlsplit(url)
        return bool(parsed.scheme and parsed.netloc)
    except Exception:
        return False

usable_url_count = non_empty_urls.map(has_scheme_and_netloc).sum()
unusable_url_count = len(non_empty_urls) - usable_url_count

print(f"Non-empty URL values: {len(non_empty_urls):,}")
print(f"Structurally usable URLs: {usable_url_count:,}")
print(f"URLs needing later review: {unusable_url_count:,}")

print("\nFirst 10 URL values needing review:")
display(
    non_empty_urls[
        ~non_empty_urls.map(has_scheme_and_netloc)
    ].head(10).to_frame(name="url")
)

Planned feature count: 28
Non-empty URL values: 253,098
Structurally usable URLs: 253,098
URLs needing later review: 0

First 10 URL values needing review:


,url


In [9]:
# Extract URL components without changing the original dataset

from urllib.parse import urlsplit

url_parts = df["url"].astype(str).map(urlsplit)

url_structure = pd.DataFrame({
    "scheme": url_parts.map(lambda part: part.scheme.lower()),
    "hostname": url_parts.map(lambda part: part.hostname),
    "path": url_parts.map(lambda part: part.path),
    "query": url_parts.map(lambda part: part.query)
})

print("URL components extracted successfully.")
display(url_structure.head())

URL components extracted successfully.


,scheme,hostname,path,query
0,https,www.visitcanada.com,,
1,http,218.228.19.9,/~yossi/9ssfpkz,
2,https,www.msupress.msu.edu,/series.php,seriesID=17
3,https,docs.google.com,/presentation/d/e/2PACX-1vRBjV4Bm4UxL3gJ8sCyQx...,start=false&amp;loop=false&amp;delayms=3000
4,https,www.c250.columbia.edu,/c250_celebrates/athletics/athletics_timeline....,


In [10]:
# Check the protocol used by each URL

scheme_counts = url_structure["scheme"].value_counts(dropna=False)

print("URL protocol distribution:")
display(scheme_counts.to_frame(name="url_count"))

print("\nURL protocol distribution percentage:")
display(
    (url_structure["scheme"].value_counts(normalize=True, dropna=False) * 100)
    .round(2)
    .to_frame(name="percentage")
)

URL protocol distribution:


,url_count
scheme,
https,182717
http,70373
ftp,8



URL protocol distribution percentage:


,percentage
scheme,
https,72.19
http,27.80
ftp,0.00


In [11]:
# Check availability of domain, path, and query-string components

domain_missing = url_structure["hostname"].isna().sum()
urls_with_path = url_structure["path"].fillna("").ne("").sum()
urls_with_query = url_structure["query"].fillna("").ne("").sum()

print(f"URLs with an extracted domain/host: {len(url_structure) - domain_missing:,}")
print(f"URLs without an extracted domain/host: {domain_missing:,}")
print(f"URLs containing a path: {urls_with_path:,}")
print(f"URLs containing a query string: {urls_with_query:,}")

URLs with an extracted domain/host: 253,098
URLs without an extracted domain/host: 0
URLs containing a path: 204,711
URLs containing a query string: 38,684


In [12]:
# Check whether the raw URLs contain patterns needed for planned feature extraction

url_text = df["url"].astype(str).str.lower()

pattern_availability = pd.DataFrame({
    "planned_pattern": [
        "Dots (.)",
        "Slashes (/)",
        "Hyphens (-)",
        "Underscores (_)",
        "Digits",
        "At symbols (@)",
        "Double slashes (//)",
        "Encoded characters (%)",
        "Punycode / IDN marker (xn--)"
    ],
    "urls_containing_pattern": [
        url_text.str.contains(r"\.", regex=True).sum(),
        url_text.str.contains("/", regex=False).sum(),
        url_text.str.contains("-", regex=False).sum(),
        url_text.str.contains("_", regex=False).sum(),
        url_text.str.contains(r"\d", regex=True).sum(),
        url_text.str.contains("@", regex=False).sum(),
        url_text.str.count("//").gt(1).sum(),
        url_text.str.contains("%", regex=False).sum(),
        url_text.str.contains("xn--", regex=False).sum()
    ]
})

display(pattern_availability)

,planned_pattern,urls_containing_pattern
0,Dots (.),253079
1,Slashes (/),253098
2,Hyphens (-),96859
3,Underscores (_),39327
4,Digits,136965
5,At symbols (@),2361
6,Double slashes (//),1610
7,Encoded characters (%),5850
8,Punycode / IDN marker (xn--),156


In [13]:
# Create a feasibility report for all 28 planned features

feature_feasibility = pd.DataFrame({
    "feature_number": list(range(1, 29)),
    "planned_feature": [
        "URL length",
        "Number of dots",
        "Number of slashes",
        "Number of hyphens",
        "Number of underscores",
        "Number of special characters",
        "Number of digits",
        "Digit-to-letter ratio",
        "URL path length",
        "Query-string length",
        "Number of query parameters",
        "Number of @ symbols",
        "Number of // occurrences",
        "URL entropy",
        "Domain length",
        "Number of subdomains",
        "TLD type",
        "Suspicious TLD",
        "Punycode/IDN usage (xn--)",
        "IP address used instead of domain",
        "HTTPS usage",
        "URL-shortening service",
        "Encoded characters (%)",
        "Hexadecimal characters",
        "Suspicious keywords",
        "Brand-name presence",
        "Excessive hyphens",
        "Random-looking domain"
    ],
    "source_available": ["url column"] * 28,
    "feasible_from_raw_url": ["Yes"] * 28,
    "implementation_status": ["To be extracted in Notebook 03"] * 28
})

display(feature_feasibility)

,feature_number,planned_feature,source_available,feasible_from_raw_url,implementation_status
0,1,URL length,url column,Yes,To be extracted in Notebook 03
1,2,Number of dots,url column,Yes,To be extracted in Notebook 03
2,3,Number of slashes,url column,Yes,To be extracted in Notebook 03
3,4,Number of hyphens,url column,Yes,To be extracted in Notebook 03
4,5,Number of underscores,url column,Yes,To be extracted in Notebook 03
5,6,Number of special characters,url column,Yes,To be extracted in Notebook 03
6,7,Number of digits,url column,Yes,To be extracted in Notebook 03
7,8,Digit-to-letter ratio,url column,Yes,To be extracted in Notebook 03
8,9,URL path length,url column,Yes,To be extracted in Notebook 03
9,10,Query-string length,url column,Yes,To be extracted in Notebook 03
